# DSA3362 Predictive Data Analytics — End-to-End Case Study

## Kaggle Telco Customer Churn

This notebook turns the main ideas in **Predictive Data Analytics** into one coherent applied workflow.

### Business problem

A telecommunications company wants to identify customers who are likely to **churn** so that retention resources can be directed toward the right customers.

The predictive target is

\[
Y =
\begin{cases}
1 & \text{customer churns}\\
0 & \text{customer stays}
\end{cases}
\]

and we learn a function

\[
\hat f(X) \approx P(Y=1\mid X).
\]

### Dataset

We use Kaggle's **Telco Customer Churn** dataset (`blastchar/telco-customer-churn`), originally based on an IBM sample dataset.

It contains roughly **7,043 customers** and 21 columns describing:

- demographics,
- subscribed services,
- account tenure,
- contract type,
- billing/payment information,
- monthly and total charges,
- churn outcome.

Kaggle dataset page:  
`https://www.kaggle.com/datasets/blastchar/telco-customer-churn`

### DSA3362 concepts illustrated

1. Predictive vs inferential modelling
2. Train/validation/test separation
3. Missing-value treatment
4. Categorical encoding and scaling
5. Data leakage prevention with `Pipeline`
6. Classification metrics
7. K-nearest neighbours
8. Decision trees
9. Bias–variance trade-off
10. Bagging / random forests
11. Gradient boosting
12. Support vector machines and kernels
13. Cross-validation
14. Hyperparameter tuning
15. ROC curves and probability thresholds
16. Permutation feature importance
17. Partial dependence
18. Principal component analysis
19. K-means clustering
20. Hierarchical clustering
21. DBSCAN
22. Exploratory factor analysis
23. Business interpretation and model limitations

> **Teaching philosophy:** each method is followed by interpretation. The goal is not merely to obtain a good AUC, but to understand *why* a model behaves as it does.

## 0. Predictive analytics mindset

A fitted model is not valuable merely because it describes the training sample well.

Our real objective is to minimize expected loss on future observations:

\[
R(f)=E[L(Y,f(X))].
\]

Because this population risk is unknown, we approximate it using resampling and held-out data.

A useful mental model is

\[
\text{Data}
\rightarrow
\text{Preprocessing}
\rightarrow
\text{Model}
\rightarrow
\text{Validation}
\rightarrow
\text{Selection}
\rightarrow
\text{Test}
\rightarrow
\text{Interpretation}.
\]

The notebook maintains three distinct roles:

- **Training set** — learn parameters and compare model families using CV.
- **Validation set** — choose probability threshold.
- **Test set** — final performance estimate.

Cross-validation is performed **inside the training data**.

## 1. Environment

The notebook uses:

- `pandas`, `numpy`
- `scikit-learn`
- `bokeh`
- optionally `kagglehub`

If packages are missing, uncomment the installation cell.

In [1]:
# %pip install -q pandas numpy scikit-learn bokeh kagglehub joblib

In [2]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Dict

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from bokeh.io import output_notebook, show
from bokeh.plotting import figure
from bokeh.models import ColumnDataSource
from bokeh.transform import factor_mark

from sklearn.base import BaseEstimator, clone
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.compose import ColumnTransformer
from sklearn.decomposition import PCA, FactorAnalysis
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
    silhouette_score,
)
from sklearn.model_selection import (
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
    cross_validate,
    train_test_split,
)
from sklearn.neighbors import KNeighborsClassifier, NearestNeighbors
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

output_notebook()

RANDOM_STATE = 42

@dataclass(frozen=True)
class Config:
    fast_mode: bool = True
    test_size: float = 0.20
    validation_fraction_of_train_val: float = 0.25
    random_state: int = RANDOM_STATE

CFG = Config()
CFG

Loading BokehJS ...

Config(fast_mode=True, test_size=0.2, validation_fraction_of_train_val=0.25, random_state=42)

### Reproducibility

`random_state=42` makes stochastic splits and estimators reproducible enough for a teaching experiment.

`fast_mode=True` keeps repeated CV and tuning moderate. Set it to `False` for slower, more stable experiments.

# Part I — Data acquisition and understanding

## 2. Robust dataset loader

The loader tries, in order:

1. a CSV beside the notebook,
2. Kaggle's conventional input directory,
3. `kagglehub`,
4. a public GitHub mirror as a final convenience fallback.

The Kaggle dataset remains the authoritative source.

In [3]:
def load_telco_churn() -> pd.DataFrame:
    filename = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

    local_candidates = [
        Path(filename),
        Path("/kaggle/input/telco-customer-churn") / filename,
    ]

    for candidate in local_candidates:
        if candidate.exists():
            print(f"Loading local dataset: {candidate}")
            return pd.read_csv(candidate)

    try:
        import kagglehub

        path = Path(kagglehub.dataset_download("blastchar/telco-customer-churn"))
        candidate = path / filename
        if candidate.exists():
            print(f"Downloaded via kagglehub: {candidate}")
            return pd.read_csv(candidate)
    except Exception as exc:
        print(f"kagglehub unavailable or download failed: {exc}")

    fallback_url = (
        "https://raw.githubusercontent.com/"
        "Rizal-A/EDA-Telco_Customer_Churn/refs/heads/main/"
        "WA_Fn-UseC_-Telco-Customer-Churn.csv"
    )
    print("Using public mirror fallback.")
    return pd.read_csv(fallback_url)


df_raw = load_telco_churn()
print("Shape:", df_raw.shape)
display(df_raw.head())

Downloaded via kagglehub: /home/anirban/.cache/kagglehub/datasets/blastchar/telco-customer-churn/versions/1/WA_Fn-UseC_-Telco-Customer-Churn.csv
Shape: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.5,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


### Expected schema

The target is `Churn`.

Useful feature groups include:

- **Demographic:** `gender`, `SeniorCitizen`, `Partner`, `Dependents`
- **Relationship:** `tenure`
- **Services:** `PhoneService`, `InternetService`, `OnlineSecurity`, ...
- **Commercial:** `Contract`, `PaymentMethod`
- **Financial:** `MonthlyCharges`, `TotalCharges`

`customerID` is an identifier, not a meaningful predictive feature.

In [4]:
df_raw.info()

display(
    pd.DataFrame({
        "dtype": df_raw.dtypes.astype(str),
        "missing": df_raw.isna().sum(),
        "n_unique": df_raw.nunique(dropna=False),
    })
)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


,dtype,missing,n_unique
customerID,object,0,7043
gender,object,0,2
SeniorCitizen,int64,0,2
Partner,object,0,2
Dependents,object,0,2
tenure,int64,0,73
PhoneService,object,0,2
MultipleLines,object,0,3
InternetService,object,0,3
OnlineSecurity,object,0,3


## 3. Data-quality issue: `TotalCharges`

A classic trap is that `TotalCharges` may be read as `object` because some rows contain blank strings.

We coerce invalid strings to missing values:

\[
\text{blank string}\rightarrow NaN.
\]

We do **not** globally impute them here because imputation statistics must be estimated from training data.

In [5]:
df = df_raw.copy()
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Missing TotalCharges after coercion:", df["TotalCharges"].isna().sum())

display(
    df.loc[
        df["TotalCharges"].isna(),
        ["customerID", "tenure", "MonthlyCharges", "TotalCharges"],
    ].head(12)
)

Missing TotalCharges after coercion: 11


,customerID,tenure,MonthlyCharges,TotalCharges
488,4472-LVYGI,0,52.55,NaN
753,3115-CZMZD,0,20.25,NaN
936,5709-LVOEQ,0,80.85,NaN
1082,4367-NUYAO,0,25.75,NaN
1340,1371-DWPAZ,0,56.05,NaN
3331,7644-OMVMY,0,19.85,NaN
3826,3213-VVOLG,0,25.35,NaN
4380,2520-SGTTA,0,20.00,NaN
5218,2923-ARZLG,0,19.70,NaN
6670,4075-WKNIU,0,73.35,NaN


## 4. Target distribution and why accuracy is insufficient

If 73% of customers stay, an algorithm predicting "stay" for everyone already obtains about 73% accuracy.

We therefore track:

\[
\text{Balanced Accuracy}
=
\frac{\text{Sensitivity}+\text{Specificity}}{2},
\]

\[
F_1=2\frac{PR}{P+R},
\]

and ROC-AUC in addition to accuracy.

In [6]:
target_counts = df["Churn"].value_counts().sort_index()
target_rate = df["Churn"].value_counts(normalize=True).sort_index()

display(pd.DataFrame({
    "count": target_counts,
    "proportion": target_rate,
}))

p = figure(
    x_range=target_counts.index.tolist(),
    height=350,
    width=650,
    title="Target distribution: customer churn",
    x_axis_label="Churn",
    y_axis_label="Customers",
)
p.vbar(
    x=target_counts.index.tolist(),
    top=target_counts.values,
    width=0.65,
)
show(p)

,count,proportion
Churn,,
No,5174,0.73463
Yes,1869,0.26537


### Interpretation

The target is moderately imbalanced rather than extremely imbalanced.

Hence:

- accuracy is useful but incomplete,
- recall tells us how many churners are found,
- precision tells us how concentrated retention actions are among actual churners,
- ROC-AUC evaluates ranking across thresholds.

## 5. Predictive EDA: contract type and churn

Predictive EDA asks:

> Which observed variables contain signal about future labels?

This is not causal inference. A strong association between contract type and churn does **not** by itself prove that changing contract type causes churn to change.

In [7]:
contract_summary = (
    df.assign(ChurnFlag=df["Churn"].map({"No": 0, "Yes": 1}))
      .groupby("Contract", as_index=False)
      .agg(
          customers=("customerID", "size"),
          churn_rate=("ChurnFlag", "mean"),
          avg_tenure=("tenure", "mean"),
      )
      .sort_values("churn_rate", ascending=False)
)

display(contract_summary)

p = figure(
    x_range=contract_summary["Contract"].tolist(),
    height=380,
    width=800,
    title="Observed churn rate by contract type",
    x_axis_label="Contract",
    y_axis_label="Churn rate",
)
p.vbar(
    x=contract_summary["Contract"].tolist(),
    top=contract_summary["churn_rate"].tolist(),
    width=0.65,
)
show(p)

,Contract,customers,churn_rate,avg_tenure
0,Month-to-month,3875,0.427097,18.036645
1,One year,1473,0.112695,42.044807
2,Two year,1695,0.028319,56.735103


## 6. Numeric distributions by churn

A predictor can be useful when

\[
P(X\mid Y=1)
\]

differs from

\[
P(X\mid Y=0).
\]

The next plots compare empirical class-conditional distributions.

In [8]:
def bokeh_hist_by_class(
    data: pd.DataFrame,
    column: str,
    bins: int = 30,
) -> None:
    p = figure(
        height=350,
        width=800,
        title=f"{column} distribution by churn",
        x_axis_label=column,
        y_axis_label="Density",
    )

    for churn_value in ["No", "Yes"]:
        values = (
            data.loc[data["Churn"] == churn_value, column]
            .dropna()
            .to_numpy()
        )
        hist, edges = np.histogram(values, bins=bins, density=True)
        centers = (edges[:-1] + edges[1:]) / 2

        p.line(
            centers,
            hist,
            line_width=2,
            legend_label=f"Churn={churn_value}",
        )

    p.legend.click_policy = "hide"
    show(p)


for col in ["tenure", "MonthlyCharges", "TotalCharges"]:
    bokeh_hist_by_class(df, col)

### What to look for

Typical useful signals include:

- shorter tenure among many churners,
- differences in monthly charges,
- strong relationship between tenure and accumulated `TotalCharges`.

These are predictive associations, not necessarily intervention effects.

# Part II — Leakage-safe supervised learning

## 7. Build \(X\) and \(y\)

We remove:

- `customerID` — identifier,
- `Churn` — target.

The outcome becomes

\[
\text{No}\mapsto0,\qquad
\text{Yes}\mapsto1.
\]

In [9]:
X = df.drop(columns=["customerID", "Churn"]).copy()
y = df["Churn"].map({"No": 0, "Yes": 1}).astype(int)

numeric_features = X.select_dtypes(include=np.number).columns.tolist()
categorical_features = X.select_dtypes(exclude=np.number).columns.tolist()

print("Numeric features:", numeric_features)
print("Categorical features:", categorical_features)
print("X shape:", X.shape)

Numeric features: ['SeniorCitizen', 'tenure', 'MonthlyCharges', 'TotalCharges']
Categorical features: ['gender', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod']
X shape: (7043, 19)


### Why `SeniorCitizen` is numeric

It is encoded as 0/1, so pandas treats it as numeric.

That is acceptable here:

- distance models can scale it,
- linear models can use the standardized indicator,
- tree models are unaffected by monotonic scaling.

Casting it to a categorical variable would also be defensible.

## 8. Train / validation / test split

We use approximately:

- 60% training,
- 20% validation,
- 20% test.

Stratification preserves the churn rate:

\[
P(Y=1)
\]

across splits.

In [10]:
X_train_val, X_test, y_train_val, y_test = train_test_split(
    X,
    y,
    test_size=CFG.test_size,
    stratify=y,
    random_state=CFG.random_state,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val,
    y_train_val,
    test_size=CFG.validation_fraction_of_train_val,
    stratify=y_train_val,
    random_state=CFG.random_state,
)

split_summary = pd.DataFrame({
    "rows": [len(X_train), len(X_val), len(X_test)],
    "churn_rate": [y_train.mean(), y_val.mean(), y_test.mean()],
}, index=["train", "validation", "test"])

display(split_summary)

,rows,churn_rate
train,4225,0.265325
validation,1409,0.265436
test,1409,0.265436


## 9. Preprocessing pipeline

### Numeric variables

Median imputation is learned from training data.

Then

\[
z_j=\frac{x_j-\mu_j}{\sigma_j}.
\]

Scaling is important for:

- KNN,
- RBF SVM,
- PCA.

### Categorical variables

We apply one-hot encoding.

### Why use a `Pipeline`?

The pipeline ensures each CV fold learns its own:

- imputers,
- means/standard deviations,
- one-hot vocabulary.

That prevents data leakage.

In [11]:
def make_one_hot_encoder():
    # Compatibility with recent and older scikit-learn releases.
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse_output=False,
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            sparse=False,
        )


numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", make_one_hot_encoder()),
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

preprocessor

ColumnTransformer(transformers=[('num',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='median')),
                                                 ('scaler', StandardScaler())]),
                                 ['SeniorCitizen', 'tenure', 'MonthlyCharges',
                                  'TotalCharges']),
                                ('cat',
                                 Pipeline(steps=[('imputer',
                                                  SimpleImputer(strategy='most_frequent')),
                                                 ('onehot',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False))]),
                                 ['gender', 'Partner', 'Dependents',
                                  'PhoneService', 'MultipleLines',
                                  'InternetService', 'OnlineSecurity',
                                  'OnlineBackup', 'DeviceProtection',
                                  'TechSupport', 'StreamingTV',
                                  'StreamingMovies', 'Contract',
                                  'PaperlessBilling', 'PaymentMethod'])])

# Part III — Core supervised models

## 10. Algorithms and their inductive biases

### Logistic regression baseline

A linear probabilistic baseline is useful even when our main interest is nonlinear methods.

### K-nearest neighbours

For a query \(x^*\),

\[
d(x_i,x^*)=
\sqrt{\sum_{j=1}^{p}(x_{ij}-x_j^*)^2}.
\]

Small \(K\) generally means lower bias and higher variance.

### Decision tree

Recursively partitions feature space. A common classification impurity is

\[
G=1-\sum_kp_k^2.
\]

### Random forest

Bagging plus random feature subsets reduces tree correlation and variance.

### Gradient boosting

Sequentially adds weak trees:

\[
F_m(x)=F_{m-1}(x)+\eta h_m(x).
\]

### RBF SVM

Uses the kernel

\[
K(x_i,x_j)=
\exp(-\gamma\|x_i-x_j\|^2)
\]

to construct nonlinear decision boundaries.

In [12]:
def make_pipeline(model: BaseEstimator) -> Pipeline:
    return Pipeline([
        ("preprocess", clone(preprocessor)),
        ("model", model),
    ])


models: Dict[str, BaseEstimator] = {
    "Logistic Regression": make_pipeline(
        LogisticRegression(max_iter=2500)
    ),
    "KNN": make_pipeline(
        KNeighborsClassifier(n_neighbors=15)
    ),
    "Decision Tree": make_pipeline(
        DecisionTreeClassifier(
            max_depth=5,
            min_samples_leaf=20,
            random_state=RANDOM_STATE,
        )
    ),
    "Random Forest": make_pipeline(
        RandomForestClassifier(
            n_estimators=300 if CFG.fast_mode else 600,
            min_samples_leaf=3,
            max_features="sqrt",
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    ),
    "Gradient Boosting": make_pipeline(
        GradientBoostingClassifier(
            n_estimators=120 if CFG.fast_mode else 220,
            learning_rate=0.05,
            max_depth=2,
            random_state=RANDOM_STATE,
        )
    ),
    "RBF SVM": make_pipeline(
        SVC(
            C=1.0,
            kernel="rbf",
            gamma="scale",
            probability=True,
            random_state=RANDOM_STATE,
        )
    ),
}

list(models)

['Logistic Regression',
 'KNN',
 'Decision Tree',
 'Random Forest',
 'Gradient Boosting',
 'RBF SVM']

## 11. Repeated stratified cross-validation

A single validation split has sampling noise.

Repeated stratified \(K\)-fold CV estimates performance more robustly:

\[
CV=
\frac{1}{RK}
\sum_{r=1}^{R}
\sum_{k=1}^{K}
L_{rk}.
\]

We compare:

- accuracy,
- balanced accuracy,
- F1,
- ROC-AUC.

The final test set remains untouched.

In [13]:
@dataclass
class ExperimentRunner:
    models: Dict[str, BaseEstimator]
    cv: object

    def evaluate(
        self,
        X: pd.DataFrame,
        y: pd.Series,
    ) -> pd.DataFrame:
        scoring = {
            "accuracy": "accuracy",
            "balanced_accuracy": "balanced_accuracy",
            "f1": "f1",
            "roc_auc": "roc_auc",
        }

        rows = []

        for name, model in self.models.items():
            print(f"Evaluating: {name}")

            scores = cross_validate(
                model,
                X,
                y,
                cv=self.cv,
                scoring=scoring,
                n_jobs=-1,
                return_train_score=False,
                error_score="raise",
            )

            rows.append({
                "model": name,
                "accuracy_mean": scores["test_accuracy"].mean(),
                "accuracy_sd": scores["test_accuracy"].std(ddof=1),
                "balanced_accuracy_mean": scores["test_balanced_accuracy"].mean(),
                "balanced_accuracy_sd": scores["test_balanced_accuracy"].std(ddof=1),
                "f1_mean": scores["test_f1"].mean(),
                "f1_sd": scores["test_f1"].std(ddof=1),
                "roc_auc_mean": scores["test_roc_auc"].mean(),
                "roc_auc_sd": scores["test_roc_auc"].std(ddof=1),
            })

        return (
            pd.DataFrame(rows)
            .sort_values(
                ["roc_auc_mean", "f1_mean"],
                ascending=False,
            )
            .reset_index(drop=True)
        )


cv = RepeatedStratifiedKFold(
    n_splits=4 if CFG.fast_mode else 5,
    n_repeats=1 if CFG.fast_mode else 4,
    random_state=RANDOM_STATE,
)

runner = ExperimentRunner(models=models, cv=cv)
benchmark_df = runner.evaluate(X_train, y_train)

display(benchmark_df.style.format(precision=4))

Evaluating: Logistic Regression
Evaluating: KNN
Evaluating: Decision Tree
Evaluating: Random Forest
Evaluating: Gradient Boosting
Evaluating: RBF SVM


,model,accuracy_mean,accuracy_sd,balanced_accuracy_mean,balanced_accuracy_sd,f1_mean,f1_sd,roc_auc_mean,roc_auc_sd
0,Gradient Boosting,0.8033,0.0121,0.7083,0.0133,0.5772,0.0227,0.8492,0.0067
1,Logistic Regression,0.8021,0.0166,0.7223,0.0226,0.5968,0.0356,0.8463,0.0089
2,Random Forest,0.8045,0.0084,0.7111,0.0169,0.5810,0.0261,0.8437,0.0068
3,KNN,0.7894,0.0085,0.7247,0.0133,0.5964,0.0192,0.8264,0.0080
4,Decision Tree,0.7856,0.0125,0.6848,0.0381,0.5328,0.0644,0.8251,0.0044
5,RBF SVM,0.8021,0.0176,0.7075,0.0231,0.5756,0.0386,0.7987,0.0053


### How to read the benchmark

The mean estimates expected predictive performance.

The fold-to-fold standard deviation describes resampling variability; it is not itself a confidence interval.

A model decision should consider more than a one-number leaderboard:

\[
\text{performance}
+
\text{stability}
+
\text{latency}
+
\text{interpretability}
+
\text{operational cost}.
\]

In [14]:
metric_plot = benchmark_df.sort_values("roc_auc_mean")

p = figure(
    y_range=metric_plot["model"].tolist(),
    height=420,
    width=850,
    title="Cross-validated ROC-AUC by model",
    x_axis_label="Mean ROC-AUC",
    y_axis_label="Model",
)
p.hbar(
    y=metric_plot["model"].tolist(),
    right=metric_plot["roc_auc_mean"].tolist(),
    height=0.55,
)
show(p)

# Part IV — Bias–variance with tree depth

## 12. Decision-tree complexity experiment

Increasing depth generally gives

\[
\text{depth}\uparrow
\Rightarrow
\text{flexibility}\uparrow
\Rightarrow
\text{training error}\downarrow.
\]

But CV performance need not continue improving.

Very shallow trees tend to underfit.  
Very deep trees tend to overfit.

In [15]:
depths = list(range(1, 16 if CFG.fast_mode else 26))
rows = []

simple_cv = RepeatedStratifiedKFold(
    n_splits=4,
    n_repeats=1 if CFG.fast_mode else 2,
    random_state=RANDOM_STATE,
)

for depth in depths:
    model = make_pipeline(
        DecisionTreeClassifier(
            max_depth=depth,
            random_state=RANDOM_STATE,
        )
    )

    scores = cross_validate(
        model,
        X_train,
        y_train,
        cv=simple_cv,
        scoring="roc_auc",
        return_train_score=True,
        n_jobs=-1,
    )

    rows.append({
        "depth": depth,
        "train_auc": scores["train_score"].mean(),
        "cv_auc": scores["test_score"].mean(),
    })

depth_df = pd.DataFrame(rows)
display(depth_df)

p = figure(
    height=400,
    width=850,
    title="Bias–variance trade-off: decision-tree depth",
    x_axis_label="Maximum tree depth",
    y_axis_label="ROC-AUC",
)
p.line(
    depth_df["depth"],
    depth_df["train_auc"],
    line_width=2,
    legend_label="Training AUC",
)
p.line(
    depth_df["depth"],
    depth_df["cv_auc"],
    line_width=2,
    legend_label="CV AUC",
)
p.scatter(
    depth_df["depth"],
    depth_df["cv_auc"],
    size=7,
)
p.legend.location = "bottom_right"
show(p)

,depth,train_auc,cv_auc
0,1,0.733650,0.733659
1,2,0.797797,0.793112
2,3,0.828920,0.817771
3,4,0.845885,0.826616
4,5,0.862828,0.821684
5,6,0.882851,0.814643
6,7,0.906186,0.796528
7,8,0.928350,0.777904
8,9,0.950082,0.755991
9,10,0.968053,0.739422


### Interpretation

A widening training–CV gap indicates increasing variance.

For squared-error regression the classic decomposition is

\[
E[(Y-\hat f(X))^2]
=
\operatorname{Bias}^2
+
\operatorname{Variance}
+
\sigma^2.
\]

ROC-AUC does not obey this exact algebra, but the conceptual bias–variance trade-off still governs model complexity.

# Part V — Hyperparameter tuning

## 13. Randomized search for a random forest

Important hyperparameters include:

- number of trees,
- maximum depth,
- minimum leaf size,
- features considered per split.

Randomized search is often more efficient than enumerating every grid combination.

In [16]:
rf_pipeline = make_pipeline(
    RandomForestClassifier(
        n_jobs=-1,
        random_state=RANDOM_STATE,
    )
)

rf_param_distributions = {
    "model__n_estimators": [200, 350, 500, 700],
    "model__max_depth": [None, 5, 8, 12, 18],
    "model__min_samples_leaf": [1, 2, 4, 8, 15],
    "model__max_features": ["sqrt", "log2", 0.5, None],
}

rf_search = RandomizedSearchCV(
    estimator=rf_pipeline,
    param_distributions=rf_param_distributions,
    n_iter=8 if CFG.fast_mode else 24,
    scoring="roc_auc",
    cv=4,
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=1,
)

rf_search.fit(X_train, y_train)

print("Best CV ROC-AUC:", rf_search.best_score_)
print("Best parameters:")
display(pd.Series(rf_search.best_params_, name="value"))

Fitting 4 folds for each of 8 candidates, totalling 32 fits
Best CV ROC-AUC: 0.8505380887576717
Best parameters:


model__n_estimators         350
model__min_samples_leaf       8
model__max_features        log2
model__max_depth           None
Name: value, dtype: object

### Hyperparameters versus learned parameters

Let \(\lambda\) denote hyperparameters and \(\theta\) learned model parameters:

\[
\theta^*(\lambda)
=
\arg\min_\theta L(\theta;\lambda).
\]

Examples:

- split thresholds inside a tree: learned parameters,
- `max_depth`: hyperparameter.

For especially rigorous model assessment, nested cross-validation can separate tuning from performance estimation. Here, the untouched test set acts as the final outer evaluation.

# Part VI — Model selection and threshold tuning

## 14. Select a model family using training CV

The model family is selected using training CV only.

The validation set remains reserved for the probability threshold.

In [17]:
best_cv_name = benchmark_df.iloc[0]["model"]
best_cv_model = clone(models[best_cv_name])

print("Best default family:", best_cv_name)
print("Tuned RF CV ROC-AUC:", rf_search.best_score_)

if rf_search.best_score_ > benchmark_df.iloc[0]["roc_auc_mean"]:
    selected_name = "Tuned Random Forest"
    selected_model = clone(rf_search.best_estimator_)
else:
    selected_name = best_cv_name
    selected_model = best_cv_model

print("Selected model:", selected_name)

Best default family: Gradient Boosting
Tuned RF CV ROC-AUC: 0.8505380887576717
Selected model: Tuned Random Forest


## 15. Tune the decision threshold on validation data

A classifier often produces

\[
\hat p_i=P(Y_i=1\mid X_i).
\]

The default rule

\[
\hat Y_i=I(\hat p_i\ge0.5)
\]

is not mandatory.

For churn, false negatives can be expensive because they represent customers who leave without intervention.

We optimize F1 on validation data as a teaching example.

In [18]:
selected_model.fit(X_train, y_train)
val_prob = selected_model.predict_proba(X_val)[:, 1]

threshold_rows = []

for threshold in np.linspace(0.05, 0.95, 91):
    pred = (val_prob >= threshold).astype(int)

    threshold_rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, pred, zero_division=0),
        "recall": recall_score(y_val, pred, zero_division=0),
        "f1": f1_score(y_val, pred, zero_division=0),
        "balanced_accuracy": balanced_accuracy_score(y_val, pred),
    })

threshold_df = pd.DataFrame(threshold_rows)

best_threshold_row = threshold_df.loc[
    threshold_df["f1"].idxmax()
]
best_threshold = float(best_threshold_row["threshold"])

print(
    "F1-optimal validation threshold:",
    round(best_threshold, 3),
)
display(best_threshold_row.to_frame("value"))

p = figure(
    height=410,
    width=850,
    title="Validation threshold trade-off",
    x_axis_label="Probability threshold",
    y_axis_label="Metric",
)

for metric in [
    "precision",
    "recall",
    "f1",
    "balanced_accuracy",
]:
    p.line(
        threshold_df["threshold"],
        threshold_df[metric],
        line_width=2,
        legend_label=metric,
    )

p.legend.click_policy = "hide"
show(p)

F1-optimal validation threshold: 0.36


,value
threshold,0.360000
precision,0.579869
recall,0.708556
f1,0.637786
balanced_accuracy,0.761524


### From prediction to decision analytics

An F1-optimal threshold is only one policy.

Suppose contacting a customer costs money and saving a churner creates value. Then threshold selection should optimize expected utility:

\[
\text{Act if}
\quad
E[\text{benefit}\mid p]
>
E[\text{cost}\mid p].
\]

This is where predictive analytics begins to become decision analytics.

## 16. Final refit and untouched test evaluation

Now we:

1. combine train + validation,
2. refit the selected model,
3. evaluate on test exactly once.

We compare threshold 0.5 with the validation-selected threshold.

In [19]:
X_fit = pd.concat([X_train, X_val], axis=0)
y_fit = pd.concat([y_train, y_val], axis=0)

final_model = clone(selected_model)
final_model.fit(X_fit, y_fit)

test_prob = final_model.predict_proba(X_test)[:, 1]

test_pred_05 = (test_prob >= 0.50).astype(int)
test_pred_tuned = (test_prob >= best_threshold).astype(int)

def classification_metrics(y_true, pred, prob):
    return {
        "accuracy": accuracy_score(y_true, pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, pred),
        "precision": precision_score(y_true, pred, zero_division=0),
        "recall": recall_score(y_true, pred, zero_division=0),
        "f1": f1_score(y_true, pred, zero_division=0),
        "roc_auc": roc_auc_score(y_true, prob),
    }

test_results = pd.DataFrame({
    "threshold_0.50": classification_metrics(
        y_test,
        test_pred_05,
        test_prob,
    ),
    f"threshold_{best_threshold:.2f}": classification_metrics(
        y_test,
        test_pred_tuned,
        test_prob,
    ),
}).T

display(test_results.style.format(precision=4))

print("Classification report at selected threshold")
print(
    classification_report(
        y_test,
        test_pred_tuned,
        target_names=["Stay", "Churn"],
    )
)

print("Confusion matrix")
display(
    pd.DataFrame(
        confusion_matrix(y_test, test_pred_tuned),
        index=["Actual stay", "Actual churn"],
        columns=["Predicted stay", "Predicted churn"],
    )
)

,accuracy,balanced_accuracy,precision,recall,f1,roc_auc
threshold_0.50,0.8041,0.7079,0.6763,0.5027,0.5767,0.8422
threshold_0.36,0.7800,0.7563,0.5690,0.7059,0.6301,0.8422


Classification report at selected threshold
              precision    recall  f1-score   support

        Stay       0.88      0.81      0.84      1035
       Churn       0.57      0.71      0.63       374

    accuracy                           0.78      1409
   macro avg       0.73      0.76      0.74      1409
weighted avg       0.80      0.78      0.79      1409

Confusion matrix


,Predicted stay,Predicted churn
Actual stay,835,200
Actual churn,110,264


## 17. ROC curves

The ROC curve plots

\[
TPR=\frac{TP}{TP+FN}
\]

against

\[
FPR=\frac{FP}{FP+TN}
\]

over every threshold.

A useful interpretation of ROC-AUC is approximately:

\[
P(
\hat p_{\text{random positive}}
>
\hat p_{\text{random negative}}
).
\]

In [20]:
roc_candidates = {
    "Logistic Regression": models["Logistic Regression"],
    "Random Forest": models["Random Forest"],
    "Gradient Boosting": models["Gradient Boosting"],
    "RBF SVM": models["RBF SVM"],
}

p = figure(
    height=500,
    width=850,
    title="Test ROC curves",
    x_axis_label="False positive rate",
    y_axis_label="True positive rate",
)

roc_table = []

for name, estimator in roc_candidates.items():
    fitted = clone(estimator).fit(X_fit, y_fit)
    prob = fitted.predict_proba(X_test)[:, 1]

    fpr, tpr, _ = roc_curve(y_test, prob)
    auc = roc_auc_score(y_test, prob)

    roc_table.append({
        "model": name,
        "test_auc": auc,
    })

    p.line(
        fpr,
        tpr,
        line_width=2,
        legend_label=f"{name} (AUC={auc:.3f})",
    )

p.line(
    [0, 1],
    [0, 1],
    line_dash="dashed",
    line_width=1,
    legend_label="Random ranking",
)

p.legend.location = "bottom_right"
p.legend.click_policy = "hide"
show(p)

display(
    pd.DataFrame(roc_table)
    .sort_values("test_auc", ascending=False)
)

,model,test_auc
2,Gradient Boosting,0.844318
0,Logistic Regression,0.841861
1,Random Forest,0.838503
3,RBF SVM,0.790560


# Part VII — Model interpretation

## 18. Permutation importance

For a feature \(X_j\):

1. score the fitted model,
2. permute \(X_j\),
3. score again,
4. measure the drop.

\[
I_j
\approx
S_{\text{baseline}}
-
S_{\text{permuted }j}.
\]

A large decrease means the model relied strongly on that feature.

In [21]:
perm = permutation_importance(
    final_model,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=8 if CFG.fast_mode else 20,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = (
    pd.DataFrame({
        "feature": X_test.columns,
        "importance_mean": perm.importances_mean,
        "importance_sd": perm.importances_std,
    })
    .sort_values("importance_mean", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(15))

top_imp = (
    importance_df
    .head(15)
    .sort_values("importance_mean")
)

p = figure(
    y_range=top_imp["feature"].tolist(),
    height=500,
    width=850,
    title="Permutation importance on test data",
    x_axis_label="Mean decrease in ROC-AUC",
)

p.hbar(
    y=top_imp["feature"].tolist(),
    right=top_imp["importance_mean"].tolist(),
    height=0.55,
)
show(p)

,feature,importance_mean,importance_sd
0,Contract,0.037531,0.003680
1,tenure,0.025988,0.002097
2,InternetService,0.014132,0.005593
3,TotalCharges,0.013562,0.001581
4,OnlineSecurity,0.003957,0.001900
5,TechSupport,0.002825,0.001232
6,PaperlessBilling,0.002231,0.001588
7,PaymentMethod,0.001894,0.001295
8,MultipleLines,0.001183,0.000517
9,StreamingMovies,0.001021,0.001098


### Interpret carefully

Permutation importance means:

> the fitted model depends on the feature for prediction.

It does **not** mean:

> the feature causally determines churn.

Correlated features can share or mask importance.

## 19. Partial-dependence-style sensitivity

For numerical feature \(X_j\),

\[
PD_j(z)
=
\frac1n
\sum_{i=1}^n
\hat f(z,x_{i,-j}).
\]

We replace \(X_j\) with \(z\) for everyone and average predicted churn probability.

This reveals model sensitivity, not causal effect.

In [22]:
def manual_partial_dependence(
    model: BaseEstimator,
    X_reference: pd.DataFrame,
    feature: str,
    grid_points: int = 30,
) -> pd.DataFrame:

    q_low, q_high = X_reference[feature].quantile([0.02, 0.98])
    grid = np.linspace(q_low, q_high, grid_points)

    rows = []

    for value in grid:
        X_temp = X_reference.copy()
        X_temp[feature] = value

        mean_prob = (
            model.predict_proba(X_temp)[:, 1]
            .mean()
        )

        rows.append({
            "value": value,
            "mean_predicted_churn": mean_prob,
        })

    return pd.DataFrame(rows)


for feature in ["tenure", "MonthlyCharges", "TotalCharges"]:
    curve = manual_partial_dependence(
        final_model,
        X_test,
        feature,
    )

    p = figure(
        height=350,
        width=800,
        title=f"Model sensitivity: {feature}",
        x_axis_label=feature,
        y_axis_label="Average predicted churn probability",
    )

    p.line(
        curve["value"],
        curve["mean_predicted_churn"],
        line_width=3,
    )
    show(p)

# Part VIII — Principal Component Analysis

## 20. PCA motivation

One-hot encoding expands the feature space.

PCA finds orthogonal directions:

\[
Z_1=w_1^\top X
\]

where \(Z_1\) maximizes variance.

Subsequent components are orthogonal:

\[
w_i^\top w_j=0,\quad i\ne j.
\]

PCA is useful for:

- visualization,
- compression,
- denoising,
- preprocessing before clustering.

### Caveat

PCA maximizes **variance**, not target prediction.

In [23]:
unsup_preprocessor = clone(preprocessor)
X_all_processed = unsup_preprocessor.fit_transform(X)

feature_names = unsup_preprocessor.get_feature_names_out()

print("Original feature count:", X.shape[1])
print("Encoded feature count:", X_all_processed.shape[1])
print("Processed matrix shape:", X_all_processed.shape)

Original feature count: 19
Encoded feature count: 45
Processed matrix shape: (7043, 45)


In [24]:
pca_full = PCA(random_state=RANDOM_STATE)
X_pca_full = pca_full.fit_transform(X_all_processed)

explained = pca_full.explained_variance_ratio_
cumulative = np.cumsum(explained)

pca_summary = pd.DataFrame({
    "component": np.arange(1, len(explained) + 1),
    "explained_variance_ratio": explained,
    "cumulative_explained_variance": cumulative,
})

display(pca_summary.head(15))

n90 = int(np.argmax(cumulative >= 0.90) + 1)
n95 = int(np.argmax(cumulative >= 0.95) + 1)

print("Components for >=90% variance:", n90)
print("Components for >=95% variance:", n95)

p = figure(
    height=430,
    width=850,
    title="PCA scree and cumulative explained variance",
    x_axis_label="Principal component",
    y_axis_label="Variance fraction",
)

p.line(
    pca_summary["component"],
    pca_summary["explained_variance_ratio"],
    line_width=2,
    legend_label="Individual explained variance",
)

p.line(
    pca_summary["component"],
    pca_summary["cumulative_explained_variance"],
    line_width=2,
    legend_label="Cumulative explained variance",
)

p.legend.location = "center_right"
show(p)

,component,explained_variance_ratio,cumulative_explained_variance
0,1,0.272155,0.272155
1,2,0.181046,0.453201
2,3,0.081322,0.534522
3,4,0.060546,0.595068
4,5,0.044659,0.639727
5,6,0.040177,0.679904
6,7,0.035197,0.715100
7,8,0.033209,0.748309
8,9,0.028984,0.777294
9,10,0.025020,0.802314


Components for >=90% variance: 15
Components for >=95% variance: 18


## 21. Two-dimensional PCA visualization

If churners and stayers overlap in PC1/PC2, supervised classification can still work.

PCA ignores \(Y\), so the directions with maximum overall variance are not guaranteed to be the most discriminative.

In [25]:
pca2 = PCA(
    n_components=2,
    random_state=RANDOM_STATE,
)

X_pca2 = pca2.fit_transform(X_all_processed)

pca_plot_df = pd.DataFrame({
    "PC1": X_pca2[:, 0],
    "PC2": X_pca2[:, 1],
    "Churn": df["Churn"].to_numpy(),
})

plot_sample = pca_plot_df.sample(
    n=min(2500, len(pca_plot_df)),
    random_state=RANDOM_STATE,
)

p = figure(
    height=500,
    width=850,
    title="First two principal components",
    x_axis_label="PC1",
    y_axis_label="PC2",
)

for label, marker in [
    ("No", "circle"),
    ("Yes", "triangle"),
]:
    subset = plot_sample[
        plot_sample["Churn"] == label
    ]

    p.scatter(
        subset["PC1"],
        subset["PC2"],
        marker=marker,
        size=6,
        alpha=0.45,
        legend_label=f"Churn={label}",
    )

p.legend.click_policy = "hide"
show(p)

# Part IX — K-means clustering

## 22. K-means objective

K-means minimizes within-cluster squared distance:

\[
\min_{C_1,\dots,C_K}
\sum_{k=1}^{K}
\sum_{x_i\in C_k}
\|x_i-\mu_k\|^2.
\]

It alternates:

1. assign each point to its closest centroid,
2. recompute centroids,
3. repeat.

We cluster a lower-dimensional PCA representation to reduce noisy high-dimensional geometry.

In [26]:
n_cluster_pcs = max(
    2,
    min(n90, 15),
)

X_cluster = X_pca_full[:, :n_cluster_pcs]

k_rows = []

for k in range(2, 9):
    km = KMeans(
        n_clusters=k,
        n_init=20,
        random_state=RANDOM_STATE,
    )

    labels = km.fit_predict(X_cluster)

    k_rows.append({
        "k": k,
        "inertia": km.inertia_,
        "silhouette": silhouette_score(
            X_cluster,
            labels,
            sample_size=min(3000, len(X_cluster)),
            random_state=RANDOM_STATE,
        ),
    })

k_eval = pd.DataFrame(k_rows)
display(k_eval)

best_k = int(
    k_eval.loc[
        k_eval["silhouette"].idxmax(),
        "k",
    ]
)

print("Silhouette-selected K:", best_k)

p = figure(
    height=380,
    width=800,
    title="K-means selection via silhouette score",
    x_axis_label="K",
    y_axis_label="Silhouette score",
)

p.line(
    k_eval["k"],
    k_eval["silhouette"],
    line_width=2,
)
p.scatter(
    k_eval["k"],
    k_eval["silhouette"],
    size=8,
)

show(p)

,k,inertia,silhouette
0,2,60355.319553,0.261080
1,3,47315.187136,0.264502
2,4,42540.678708,0.259264
3,5,39697.585541,0.218800
4,6,37616.419926,0.220775
5,7,35720.831622,0.212251
6,8,34119.275601,0.175748


Silhouette-selected K: 3


### Silhouette score

For observation \(i\):

- \(a(i)\): mean distance to its own cluster,
- \(b(i)\): smallest mean distance to another cluster.

\[
s(i)
=
\frac{b(i)-a(i)}
{\max(a(i),b(i))}.
\]

Near 1 = well separated.  
Near 0 = overlap.  
Negative = possible misassignment.

A high silhouette score does not guarantee commercially useful segmentation.

In [27]:
kmeans = KMeans(
    n_clusters=best_k,
    n_init=30,
    random_state=RANDOM_STATE,
)

cluster_labels = kmeans.fit_predict(X_cluster)

cluster_plot_df = pca_plot_df.copy()
cluster_plot_df["cluster"] = cluster_labels.astype(str)

plot_sample = cluster_plot_df.sample(
    n=min(3000, len(cluster_plot_df)),
    random_state=RANDOM_STATE,
)

markers = [
    "circle",
    "triangle",
    "square",
    "diamond",
    "inverted_triangle",
    "hex",
    "star",
    "plus",
]

factors = sorted(
    plot_sample["cluster"].unique().tolist()
)

source = ColumnDataSource(plot_sample)

p = figure(
    height=500,
    width=850,
    title="K-means customer segments in PCA space",
    x_axis_label="PC1",
    y_axis_label="PC2",
)

p.scatter(
    x="PC1",
    y="PC2",
    source=source,
    marker=factor_mark(
        "cluster",
        markers=markers[:len(factors)],
        factors=factors,
    ),
    size=7,
    alpha=0.5,
    legend_field="cluster",
)

p.legend.title = "Cluster"
p.legend.click_policy = "hide"
show(p)

## 23. Cluster profiling

Clustering is useful only when groups can be characterized.

We did **not** use churn to create clusters. We inspect churn afterward to see whether unsupervised customer structure correlates with risk.

In [28]:
profile_df = df.copy()
profile_df["ChurnFlag"] = profile_df["Churn"].map({
    "No": 0,
    "Yes": 1,
})
profile_df["cluster"] = cluster_labels

cluster_profile = (
    profile_df
    .groupby("cluster")
    .agg(
        customers=("customerID", "size"),
        churn_rate=("ChurnFlag", "mean"),
        mean_tenure=("tenure", "mean"),
        mean_monthly_charges=("MonthlyCharges", "mean"),
        mean_total_charges=("TotalCharges", "mean"),
    )
    .sort_values("churn_rate", ascending=False)
)

display(
    cluster_profile.style.format({
        "churn_rate": "{:.3f}",
        "mean_tenure": "{:.1f}",
        "mean_monthly_charges": "{:.2f}",
        "mean_total_charges": "{:.2f}",
    })
)

,customers,churn_rate,mean_tenure,mean_monthly_charges,mean_total_charges
cluster,,,,,
1,3182,0.441,15.3,67.84,1016.89
0,2335,0.151,56.9,89.12,5059.69
2,1526,0.074,30.5,21.08,665.22


# Part X — Hierarchical clustering

## 24. Agglomerative clustering

Agglomerative clustering starts with \(n\) singleton clusters:

\[
n\rightarrow n-1\rightarrow\cdots\rightarrow1.
\]

Ward linkage merges the pair that produces the smallest increase in within-cluster variance.

Because hierarchical clustering has expensive \(O(n^2)\)-class scaling, we demonstrate it on a sample.

In [29]:
rng = np.random.default_rng(RANDOM_STATE)

sample_idx = rng.choice(
    len(X_cluster),
    size=min(1500, len(X_cluster)),
    replace=False,
)

X_hier = X_cluster[
    sample_idx,
    :min(8, X_cluster.shape[1]),
]

agg = AgglomerativeClustering(
    n_clusters=best_k,
    linkage="ward",
)

agg_labels = agg.fit_predict(X_hier)

agg_plot = pd.DataFrame({
    "PC1": X_pca2[sample_idx, 0],
    "PC2": X_pca2[sample_idx, 1],
    "cluster": agg_labels.astype(str),
})

display(
    agg_plot["cluster"]
    .value_counts()
    .sort_index()
    .rename("customers")
)

factors = sorted(
    agg_plot["cluster"].unique().tolist()
)

source = ColumnDataSource(agg_plot)

p = figure(
    height=480,
    width=820,
    title="Agglomerative clustering in PCA space",
    x_axis_label="PC1",
    y_axis_label="PC2",
)

p.scatter(
    x="PC1",
    y="PC2",
    source=source,
    marker=factor_mark(
        "cluster",
        markers=markers[:len(factors)],
        factors=factors,
    ),
    size=7,
    alpha=0.5,
    legend_field="cluster",
)

p.legend.click_policy = "hide"
show(p)

cluster
0    639
1    313
2    548
Name: customers, dtype: int64

# Part XI — DBSCAN

## 25. Density-based clustering

DBSCAN uses:

- \(\epsilon\): neighbourhood radius,
- `min_samples`: local density requirement.

Points become:

- core,
- border,
- noise.

Unlike K-means, DBSCAN can:

- discover irregularly shaped clusters,
- label noise explicitly,
- avoid specifying \(K\).

It is highly sensitive to scaling and \(\epsilon\).

In [30]:
X_db = StandardScaler().fit_transform(
    X_pca_full[
        :,
        :min(5, X_pca_full.shape[1]),
    ]
)

nn = NearestNeighbors(n_neighbors=10)
nn.fit(X_db)

distances, _ = nn.kneighbors(X_db)
k_distance = np.sort(distances[:, -1])

p = figure(
    height=380,
    width=820,
    title="DBSCAN diagnostic: sorted 10-NN distance",
    x_axis_label="Sorted observation",
    y_axis_label="10-NN distance",
)

p.line(
    np.arange(len(k_distance)),
    k_distance,
    line_width=2,
)
show(p)

### Choosing \(\epsilon\)

A common heuristic looks for the knee in the sorted k-nearest-neighbour distance curve.

The next cell uses a starting value. Experiment with it rather than treating it as universally optimal.

In [31]:
dbscan = DBSCAN(
    eps=0.9,
    min_samples=15,
)

db_labels = dbscan.fit_predict(X_db)

n_clusters_db = len(
    set(db_labels) - {-1}
)
n_noise = int(
    np.sum(db_labels == -1)
)

print("DBSCAN clusters:", n_clusters_db)
print("Noise points:", n_noise)
print("Noise fraction:", n_noise / len(db_labels))

display(
    pd.Series(db_labels)
    .value_counts()
    .sort_index()
    .rename("count")
)

DBSCAN clusters: 7
Noise points: 17
Noise fraction: 0.002413744143120829


-1      17
 0    4427
 1     646
 2    1077
 3     305
 4     523
 5      23
 6      25
Name: count, dtype: int64

If nearly everything is noise or nearly everything forms one cluster, the code is not necessarily wrong. It means the chosen density scale is not revealing useful structure.

Try:

```python
for eps in [0.5, 0.7, 0.9, 1.1, 1.3]:
    ...
```

and inspect cluster count, noise fraction and stability.

# Part XII — Exploratory factor analysis

## 26. PCA versus factor analysis

### PCA

\[
Z=W^\top X
\]

constructs components that explain variance.

### Factor analysis

\[
X=\Lambda F+\epsilon
\]

assumes observed variables are generated by latent factors plus variable-specific noise.

We demonstrate it on service-engagement indicators.

> **Caveat:** these indicators are binary, so classical Gaussian factor analysis is only an approximate teaching demonstration. Tetrachoric/polychoric approaches may be preferable for rigorous binary-item factor models.

In [32]:
service_cols = [
    "PhoneService",
    "MultipleLines",
    "OnlineSecurity",
    "OnlineBackup",
    "DeviceProtection",
    "TechSupport",
    "StreamingTV",
    "StreamingMovies",
    "Partner",
    "Dependents",
    "PaperlessBilling",
]

service_matrix = pd.DataFrame(index=df.index)

for col in service_cols:
    service_matrix[col] = (
        df[col] == "Yes"
    ).astype(float)

service_scaled = StandardScaler().fit_transform(
    service_matrix
)

fa = FactorAnalysis(
    n_components=3,
    random_state=RANDOM_STATE,
)

fa.fit(service_scaled)

loadings = pd.DataFrame(
    fa.components_.T,
    index=service_cols,
    columns=[
        "Factor 1",
        "Factor 2",
        "Factor 3",
    ],
)

display(loadings.round(3))

,Factor 1,Factor 2,Factor 3
PhoneService,-0.031,0.002,-0.478
MultipleLines,0.393,-0.062,-0.446
OnlineSecurity,0.399,0.083,0.253
OnlineBackup,0.484,-0.015,0.090
DeviceProtection,0.603,-0.061,0.118
TechSupport,0.491,0.016,0.271
StreamingTV,0.656,-0.167,-0.060
StreamingMovies,0.659,-0.187,-0.046
Partner,0.332,0.615,-0.110
Dependents,0.109,0.666,0.003


### Reading loadings

Large positive or negative loadings indicate strong association with a latent factor.

Possible interpretations might involve:

- household relationship,
- service breadth,
- online-service adoption.

Do not force semantic labels if the loading pattern does not support them.

# Part XIII — Algorithmic complexity

## 27. Approximate time and space complexity

Let:

- \(n\) = training observations
- \(p\) = encoded features
- \(B\) = number of trees
- \(T\) = boosting rounds
- \(k\) = number of clusters
- \(I\) = K-means iterations

These are order-of-growth approximations.

| Method | Typical training cost | Prediction / main cost | Engineering implication |
|---|---:|---:|---|
| KNN | \(\Theta(np)\) storage / almost no fit | \(\Theta(np)\) per brute-force query | Expensive inference |
| Decision tree | roughly \(O(np\log n)\) | \(O(\text{depth})\) | Fast inference |
| Random forest | roughly \(O(Bnp\log n)\) | \(O(B\cdot\text{depth})\) | Easy to parallelize |
| Gradient boosting | roughly \(O(Tnp\log n)\) | \(O(T\cdot\text{depth})\) | Sequential training |
| Kernel SVM | often \(O(n^2p)\) to \(O(n^3)\) | depends on support vectors | Poor scaling for huge \(n\) |
| K-means | \(O(nkpI)\) | \(O(kp)\) assignment | Linear in \(n\) per iteration |
| PCA / SVD | roughly \(O(\min(np^2,n^2p))\) | matrix projection | Randomized SVD helps |
| Agglomerative clustering | often \(O(n^2)\)-class | expensive clustering | Sampling may be needed |

A slightly higher AUC may not justify a model with much higher latency, memory usage or retraining cost.

# Part XIV — Optional production artifact

## 28. Serialize the full pipeline

Because preprocessing is inside the fitted pipeline, serializing `final_model` preserves:

- imputation,
- scaling,
- one-hot categories,
- predictive estimator.

That is safer than serializing the estimator alone.

In [33]:
# Optional:
#
# import joblib
#
# joblib.dump(
#     {
#         "model": final_model,
#         "threshold": best_threshold,
#         "feature_columns": X.columns.tolist(),
#     },
#     "telco_churn_pipeline.joblib",
# )

# Part XV — Exercises with solutions

## Exercise 1 — Metric choice

Suppose only 5% of customers churn and a model predicts "stay" for everyone.

1. What accuracy does it obtain?
2. Why is this inadequate?
3. Which other metrics would you inspect?

<details>
<summary><b>Solution</b></summary>

1. Accuracy = 95%.
2. Churn recall = 0, so the model completely fails the churn-detection objective.
3. Inspect recall, precision, F1, balanced accuracy, ROC-AUC and especially PR-AUC under severe imbalance.

</details>

## Exercise 2 — Leakage

Fit preprocessing once on the full dataset before CV.

Why would the resulting CV estimate be optimistic?

<details>
<summary><b>Solution</b></summary>

The held-out fold influences imputation statistics, scaling parameters and category discovery.

Correct design:

\[
\text{fit preprocessing inside each training fold}.
\]

</details>

## Exercise 3 — KNN and scaling

Train KNN:

1. with standardization,
2. without standardization.

Explain the difference using

\[
d(x_i,x_j)
=
\sqrt{\sum_r(x_{ir}-x_{jr})^2}.
\]

<details>
<summary><b>Solution</b></summary>

Without scaling, large-range numeric variables dominate Euclidean distance, effectively downweighting smaller-scale dimensions.

</details>

## Exercise 4 — Bias and variance

Extend the decision-tree depth experiment to depth 30.

Find a region where:

- training AUC rises,
- CV AUC stagnates or falls.

Explain the widening gap.

<details>
<summary><b>Solution</b></summary>

The deeper tree fits increasingly sample-specific partitions. Training error decreases, but sensitivity to sample noise increases. This is increasing variance / overfitting.

</details>

## Exercise 5 — Economic threshold

Assume:

- contacting a customer costs \$10,
- retaining a true churner produces \$100 value,
- outreach succeeds 25% of the time.

For each validation threshold compute:

\[
\text{profit}
=
TP(0.25\times100)
-
(TP+FP)(10).
\]

Choose the threshold maximizing validation profit.

<details>
<summary><b>Solution sketch</b></summary>

```python
pred = (val_prob >= threshold).astype(int)
tn, fp, fn, tp = confusion_matrix(y_val, pred).ravel()
profit = tp * 25 - (tp + fp) * 10
```

Fix the selected threshold, then evaluate the policy on test data.

</details>

## Exercise 6 — PCA

How many PCs explain:

- 80%,
- 90%,
- 95%

of variance?

Would you automatically choose 95% before supervised prediction?

<details>
<summary><b>Solution</b></summary>

No. PCA preserves variance, not necessarily target information. If the goal is prediction, choose dimensionality using supervised validation performance.

</details>

## Exercise 7 — Cluster stability

Repeat K-means with multiple random seeds.

Compare:

- inertia,
- silhouette,
- cluster size,
- churn rate by cluster.

Why is unstable segmentation operationally dangerous?

<details>
<summary><b>Solution</b></summary>

If tiny initialization changes substantially alter segment assignments, downstream marketing or retention actions become inconsistent. Operational segments should be interpretable and reasonably stable.

</details>

## Exercise 8 — SVM hyperparameters

Explore

\[
C\in\{0.1,1,10\}
\]

and

\[
\gamma\in\{\text{scale},0.01,0.1\}.
\]

Interpret:

- larger \(C\): stronger penalty for margin violations,
- larger \(\gamma\): more localized RBF influence and more flexible boundaries.

Predict which combinations are most likely to overfit before running them.

# Part XVI — Further experiment scaffold

In [34]:
experiments = {
    "KNN small K": make_pipeline(
        KNeighborsClassifier(n_neighbors=3)
    ),
    "KNN smoother": make_pipeline(
        KNeighborsClassifier(n_neighbors=35)
    ),
    "Tree shallow": make_pipeline(
        DecisionTreeClassifier(
            max_depth=2,
            random_state=RANDOM_STATE,
        )
    ),
    "Tree deep": make_pipeline(
        DecisionTreeClassifier(
            max_depth=None,
            random_state=RANDOM_STATE,
        )
    ),
    "SVM low C": make_pipeline(
        SVC(
            C=0.1,
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
        )
    ),
    "SVM high C": make_pipeline(
        SVC(
            C=10,
            kernel="rbf",
            probability=True,
            random_state=RANDOM_STATE,
        )
    ),
}

experiment_runner = ExperimentRunner(
    models=experiments,
    cv=RepeatedStratifiedKFold(
        n_splits=4,
        n_repeats=1,
        random_state=RANDOM_STATE,
    ),
)

# Uncomment:
# display(
#     experiment_runner.evaluate(
#         X_train,
#         y_train,
#     )
# )

# Part XVII — What this case study teaches

## 1. Generalisation beats training fit

\[
\text{low training error}
\not\Rightarrow
\text{good prediction}.
\]

Cross-validation and held-out testing estimate generalisation.

## 2. Complexity must be validated

For KNN, trees, SVMs and boosting:

\[
\text{more flexible}
\not\Rightarrow
\text{better}.
\]

The useful complexity level balances bias and variance.

## 3. Preprocessing is part of the model

Imputation, scaling, encoding and PCA must obey train/test boundaries.

## 4. Probabilities are richer than labels

\[
P(\text{churn})=0.72
\]

contains more decision information than a bare `Churn` prediction.

## 5. Predictive importance is not causal effect

A variable may forecast churn without being a valid intervention target.

## 6. Unsupervised learning answers different questions

Supervised learning:

\[
X\rightarrow Y?
\]

Clustering:

\[
\text{What structure exists in }X?
\]

PCA:

\[
\text{Which lower-dimensional directions explain variation in }X?
\]

Factor analysis:

\[
\text{Which latent factors may generate observed covariance?}
\]

---

# Final mental model

\[
\boxed{
\text{Predictive Analytics}
=
\text{Representation}
+
\text{Learning}
+
\text{Validation}
+
\text{Decision}
+
\text{Interpretation}
}
\]

A strong predictive-data scientist does more than run algorithms. They control leakage, understand bias–variance, select metrics aligned to the objective, validate complexity, distinguish prediction from causality, and connect model outputs to decisions.

## References and advanced extensions

Recommended companions:

- NUS DSA3362 — Predictive Data Analytics
- James, Witten, Hastie, Tibshirani & Taylor — *An Introduction to Statistical Learning*
- Hastie, Tibshirani & Friedman — *The Elements of Statistical Learning*
- scikit-learn user guide
- Kaggle — Telco Customer Churn

### Advanced extensions

1. nested cross-validation,
2. calibration and Brier score,
3. precision–recall curves,
4. SHAP,
5. XGBoost / LightGBM / CatBoost,
6. class-weighted learning,
7. cost-sensitive learning,
8. survival analysis for time-to-churn,
9. conformal prediction,
10. drift monitoring and production MLOps.